# Adaptive scans: a strategy portfolio

`escape.adaptive_scan` decides **where to measure next, with which exposure, and in which order**. Every point costs
time: moves, settling, backlash correction and statistics. All strategies share one planner, which maximises
*expected value per second*. They differ in what they assume about the signal:

| strategy | assumes | use when |
|---|---|---|
| `UniformScan` | nothing | baseline |
| `ModelFreeScan` / `GradientScan` | smooth curve | unknown signals / edges |
| `StepScan` | an edge | time zero, knife edges |
| `OscillationScan` | step + 1–3 damped, growing-in oscillations | impulsive responses |
| `PeakScan2D` | tilted, asymmetric 2-D peak | beam profiles, 2-D maps |

See `docs/user_guide/adaptive_scan.md` for the methods.

In [ ]:
import _devpath; _devpath.prepend_local_checkout_to_path()
import numpy as np
import matplotlib.pyplot as plt
from escape import adaptive_scan as ads
from escape.adaptive_scan import ImpulsiveResponse, Oscillation

## 1. Costs: motion with backlash, exposure, visiting order

The delay stage below needs 0.5 s per move plus 1 s per unit of travel. It approaches positions from below, so going
backwards costs an overshoot and a second approach. Exposures can be chosen from ½× to 8× the base.

In [ ]:
stage = ads.Axis(speed=1.0, overhead=0.5, backlash=0.05, direction=+1)
print("move 0 → 1:", stage.move_time(0, [1.0])[0], "s;   1 → 0 (backlash):", round(stage.move_time(1, [0.0])[0], 2), "s")

# the fastest order to visit a batch, starting at 2.5
pts = np.random.default_rng(1).uniform(0, 5, (8, 1))
order, t = ads.order_route(pts, [2.5], ads.Motion(stage))
print("as given:", round(ads.route_time(pts, [2.5], ads.Motion(stage)), 1), "s   optimal order:", round(t, 1), "s")
print("visit:", np.round(pts[order, 0], 2))

# costs can be learnt from logged moves
rng = np.random.default_rng(0); a, b = rng.uniform(0, 5, 100), rng.uniform(0, 5, 100)
log = [stage.move_time(x, [y])[0] + 0.02 * rng.standard_normal() for x, y in zip(a, b)]
print(ads.Axis.calibrate(a, b, log))

## 2. Model-free: an unknown signal over a wide range

Here the signal is the default damped oscillation, scanned over (−2, 15) on a 0.01 grid, with the stage above and
automatic exposure choice. Both strategies get the same 300 s of simulated beamtime.

In [ ]:
truth = ImpulsiveResponse.default()
fig, axs = plt.subplots(2, 2, figsize=(13, 6), sharex=True, height_ratios=(3, 1))
for col, (name, make) in enumerate([("uniform", ads.UniformScan), ("model-free", ads.ModelFreeScan)]):
    sim = ads.GaussianSimulator(truth, noise=0.08, sigma=0.05, seed=4)
    instr = ads.Instrument(sim, motion=stage)
    scan = make((-2, 15), grid_step=0.01, motion=stage, exposure=ads.ExposureModel("auto"))
    ads.run_plan(scan, instr, budget=300, batch=5)
    g = ads.ModelFreeScan((-2, 15)); g.add(scan.x, scan.y, scan.yerr)
    g.plot(axes=axs[:, col], truth=sim.truth)
    _i, x, y, e = g.averaged(); tt = np.linspace(-2, 15, 4000)
    rms = np.sqrt(np.mean((np.interp(tt, x, y) - sim.truth(tt)) ** 2))
    axs[0, col].set_title(f"{name}: {scan.n_points} points, {instr.move_time:.0f} s moving, "
                          f"{instr.n_reversals} reversals, RMS {rms:.3f}")

## 3. Step search

Here a step sits at an unknown position in (0, 20). `StepScan` keeps the plausible step positions as competing
hypotheses and measures where they disagree, which works like a noise-aware bisection. Then it refines the edge.

In [ ]:
step_truth = ImpulsiveResponse([Oscillation(1.0)], t0=12.34)
sim = ads.GaussianSimulator(step_truth, noise=0.3, sigma=0.08, seed=2)
instr = ads.Instrument(sim, motion=stage)
step = ads.StepScan((0, 20), grid_step=0.01, motion=stage, exposure=ads.ExposureModel("auto"))
ads.run_plan(step, instr, budget=150, batch=4)
print(step.summary()); print(f"(truth t0 = 12.34, sigma = 0.08)   time {instr.clock:.0f} s")
fig, ax = plt.subplots(figsize=(10, 3))
ax.errorbar(step.x, step.y, step.yerr, fmt=".", ms=3, lw=0.6)
ax.plot(step.candidates, step.best.predict(step.candidates))
ax.set_xlabel("position"); ax.set_title("StepScan: survey, then points concentrate on the edge");

## 4. Oscillations: the default signal, then 3 overlapping modes with own rise and decay times

The single damped oscillation uses the realistic pump-probe simulator (I0 fluctuations, jitter, drift; 100 shots per
point). The shot-by-shot simulator has no motion model here.

In [ ]:
measure = ads.PumpProbeSimulator(shots_per_point=100, seed=3)
scan = ads.OscillationScan((-2, 15), max_osc=2)
ads.run_scan(scan, measure, n_initial=15, n_rounds=30, points_per_round=5)
print(scan.summary())
axes = scan.plot(truth=measure.truth)

In [ ]:
three = ImpulsiveResponse([Oscillation(1.0, decay=8.0),
                           Oscillation(-0.5, period=0.7, decay=2.0),
                           Oscillation(0.35, period=0.31, decay=1.2, rise=0.4),
                           Oscillation(0.25, period=1.9, decay=4.0, rise=1.0)])
sim = ads.GaussianSimulator(three, noise=0.03, sigma=0.03, seed=0)
instr = ads.Instrument(sim, motion=stage)
osc3 = ads.OscillationScan((-1, 8), max_osc=3, fit_rise=True, grid_step=0.01, motion=stage)
ads.run_plan(osc3, instr, budget=900, batch=8, n_initial=40)
print(osc3.summary())
print("truth: periods 0.31 / 0.7 / 1.9, decays 1.2 / 2 / 4, rises 0.4 / 0 / 1.0")
axes = osc3.plot(truth=sim.truth)

## 5. 2-D peak: size, tilt and asymmetry

Here a tilted (30°), asymmetric peak is scanned with two axes. The y axis is half as fast as x, and both have backlash.
`PeakScan2D` keeps a symmetric and an asymmetric hypothesis. The second panel shows where the next points would go.

In [ ]:
peak = ads.Peak2D(x0=0.7, y0=-0.4, sigma_u=(0.35, 0.6), sigma_v=0.2, angle=np.deg2rad(30), amplitude=1.0, offset=0.05)
xy = ads.Motion([ads.Axis(speed=1.0, overhead=0.3, backlash=0.05), ads.Axis(speed=0.5, overhead=0.3, backlash=0.05)])
sim = ads.GaussianSimulator(peak, noise=0.05, seed=3)
instr = ads.Instrument(sim, motion=xy, ndim=2)
p2 = ads.PeakScan2D(((-3, 3), (-3, 3)), grid_step=0.05, motion=xy)
ads.run_plan(p2, instr, budget=300, batch=6)
print(p2.summary())
print("truth: fwhm", np.round(peak.fwhm, 3), " angle 30°, asym_u", round((0.6 - 0.35) / 0.95, 3))
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
p2.plot(ax=axs[0], truth=peak); p2.plot(ax=axs[1], show="gain")
axs[0].set_title(f"{p2.n_points} points in {instr.clock:.0f} s"); axs[1].set_title("information of the next point");

## 6. Benchmarks

See the tables in `docs/user_guide/adaptive_scan.md` (equal simulated beamtime, including moves and backlash).